# Verify whether requests use the increased quota

In [ ]:
# ==========================================
# Step 1: Install the current official unified SDK (2026).
# ==========================================
print("📦 正在安装最新版 google-genai 库...")
!pip install google-genai -q
print("✅ 安装完成！\n")

# ==========================================
# Step 2: Obtain the underlying Application Default Credentials (ADC) with the necessary permissions.
# ==========================================
print("🔑 准备进行底层授权...")
print("⚠️ 请注意：下方会生成一个蓝色链接。请点击它，并使用带有 300 刀赠金的 Google 账号登录授权。")
print("⚠️ 授权后复制验证码，粘贴回下方的输入框中并回车。")
!gcloud auth application-default login

# ==========================================
# Step 3: Assign the new project as the quota and billing project.
# ==========================================
print("\n💰 正在将计费接口挂载到你的新项目 [api-try-495100]...")
!gcloud auth application-default set-quota-project api-try-495100
print("✅ 配额注入成功！\n")

# ==========================================
# Step 4: Call Gemini 2.5 Flash.
# ==========================================
from google import genai

print("🔄 正在连接到自由的新世界...")

try:
    # Set vertexai=True to route requests through Vertex AI.
    client = genai.Client(
        vertexai=True,
        project="api-try-495100",
        location="us-central1"
    )

    print("🚀 正在使用 gemini-2.5-flash 模型发送请求...")
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="你好，总指挥！这是从全新的无组织专属项目中发出的第一条测试消息。如果你收到了，请用极其热烈的语气恭喜我'金蝉脱壳'彻底成功！"
    )

    print("\n" + "="*40)
    print("🎉🎉🎉 彻底成功！模型回复如下：\n")
    print(response.text)
    print("="*40 + "\n")
    print("💡 下一步：你现在可以放心地把这个初始化逻辑，无缝接入到你的 CSV 批处理循环中了！")

except Exception as e:
    print("\n❌ 报错了，请检查信息：", e)

# Initialisation

In [ ]:
# -*- coding: utf-8 -*-

import json
import time
import random
import hashlib
import sqlite3
import re
import requests
from datetime import datetime
from pathlib import Path
import pandas as pd

In [ ]:
import sys
from google.colab import drive

# 1. Mount Google Drive; if it is not mounted, an authorisation dialog appears.
try:
    if not Path('/content/drive').exists():
        drive.mount('/content/drive')
    print("✅ Drive 状态: 已挂载")
except Exception as e:
    print(f"❌ Drive 挂载失败: {e}")

# 2. Check that the key variables have been defined.
required_vars = ['INPUT_FILE', 'SYSTEM', 'call_api', 'read_csv_robust']
missing = [v for v in required_vars if v not in globals()]
if missing:
    print(f"⚠️ 警告: 缺少关键定义 {missing}。请确保运行了上方所有 cell_id 为 new_cell_x 的单元格。")
else:
    print(f"🚀 准备就绪，开始处理文件: {INPUT_FILE}")
    # 3. Run the main workflow manually using the code in step 10.
    try:
        # Call the previously defined logic to label records in a loop.
        # If the main workflow has not run yet, run the cell identified by cell_id: new_cell_20 below.
        print("请点击运行下方的 '⑩ 主流程' 单元格来开始打标工作。")
    except NameError:
        print("请先从上到下依次点击每个代码块左侧的播放按钮。")

### 10. Main workflow

In [ ]:
from pathlib import Path
import re
import pandas as pd
from datetime import datetime

INPUT_BASE_DIR = Path(r"/content/drive/MyDrive/CES分析/源数据-重分类/香山-植物园")
OUTPUT_BASE_DIR = Path(r"/content/drive/MyDrive/CES分析/评论-舒适度分析结果香山-植物园")

# Helper to read CSV robustly (copied from main processing cell)
def read_csv_robust(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        return pd.read_csv(path, encoding="gbk")

# Helper to extract timestamp from filename for sorting (copied from main processing cell)
def get_timestamp_from_filename(filepath: Path) -> datetime:
    match = re.search(r'-(\d{8}_\d{4})\.csv$', filepath.name)
    if match:
        return datetime.strptime(match.group(1), '%Y%m%d_%H%M')
    return datetime.min # Return smallest possible datetime if no timestamp found

def make_comment_id(text: str): # Moved from new_cell_14 to be available for validation
    return hashlib.md5(text.encode("utf-8", errors="ignore")).hexdigest()

def guess_text_col(df: pd.DataFrame):
    candidates = ["content", "评论", "comment", "review", "text", "内容", "评价", "点评内容", "comment_detail"]
    for c in candidates:
        if c in df.columns:
            return c
    obj_cols = [c for c in df.columns if df[c].dtype == "object"]
    if not obj_cols:
        return None
    lens = {c: df[c].astype(str).map(len).mean() for c in obj_cols}
    return max(lens, key=lens.get)


# Dictionary to store input file base name -> full path -> set of expected comment_ids
input_files_expected_cids = {}
for file_path in INPUT_BASE_DIR.glob('**/*.csv'):
    if file_path.parent == INPUT_BASE_DIR:
        base_name = file_path.stem
    else:
        base_name = file_path.parent.name

    try:
        df_input = read_csv_robust(file_path)
        text_col = guess_text_col(df_input)
        if text_col is None:
            print(f"警告: 输入文件 {file_path.name} 未找到合适的文本列，跳过此文件进行完整性检查。")
            continue

        expected_cids = set()
        for text_content in df_input[text_col].dropna().astype(str):
            if text_content.strip():
                expected_cids.add(make_comment_id(text_content.strip()))

        input_files_expected_cids[base_name] = {"path": file_path, "expected_cids": expected_cids}

    except Exception as e:
        print(f"警告: 无法读取输入文件 {file_path.name} ({e})，将跳过此文件进行完整性检查。")


# Dictionary to store output file base name -> latest output path and its timestamp and actual cids
output_latest_files = {}
for file_path in OUTPUT_BASE_DIR.glob('*.csv'):
    match = re.search(r"^(.*?)-\d{8}_\d{4}\.csv$", file_path.name)
    base_name = match.group(1) if match else file_path.stem

    current_timestamp = get_timestamp_from_filename(file_path)

    if base_name not in output_latest_files or current_timestamp > output_latest_files[base_name]["timestamp"]:
        output_latest_files[base_name] = {"path": file_path, "timestamp": current_timestamp, "actual_cids": set()} # Initialize actual_cids

# Now, read the comment_ids for the latest output files
for base_name, info in output_latest_files.items():
    try:
        df_output = read_csv_robust(info["path"])
        if "comment_id" in df_output.columns:
            output_latest_files[base_name]["actual_cids"] = set(df_output["comment_id"].dropna().astype(str).tolist())
        else:
            print(f"警告: 输出文件 {info['path'].name} 中未找到 'comment_id' 列，无法进行精确完整性检查。")
            output_latest_files[base_name]["actual_cids"] = set() # Treat as empty if comment_id not found

    except Exception as e:
        print(f"警告: 无法读取输出文件 {info['path'].name} ({e})，该文件将标记为未完全处理。")
        output_latest_files[base_name]["actual_cids"] = set() # If read fails, assume no CIDs


missing_output_files = []
incomplete_processing_files = []
processed_complete_files = []

for input_base_name, input_data in input_files_expected_cids.items():
    expected_cids = input_data["expected_cids"]

    if input_base_name not in output_latest_files:
        if expected_cids: # Only report missing if there are actual comments to process
            missing_output_files.append(input_base_name)
        else: # If input has no valid comments, consider it complete if no output needed
            processed_complete_files.append(input_base_name + " (无有效评论)")

    else:
        output_data = output_latest_files[input_base_name]
        actual_cids = output_data["actual_cids"]

        # Check if all expected comment_ids are present in the actual output comment_ids
        if not expected_cids.issubset(actual_cids): # If actual_cids doesn't contain all expected_cids
            missing_count = len(expected_cids - actual_cids)
            incomplete_processing_files.append(
                f"{input_base_name} (缺失 {missing_count} 条评论ID；输入预期总数: {len(expected_cids)}, 输出实际总数: {len(actual_cids)} - {output_data['path'].name})")
        else:
            processed_complete_files.append(input_base_name)

if not missing_output_files and not incomplete_processing_files:
    print("✅ 所有输入文件都在输出文件夹中找到了对应的结果文件，且评论ID匹配。")
    print(f"共计 {len(processed_complete_files)} 个文件已完全处理。")
else:
    if missing_output_files:
        print("❌ 以下输入文件尚未在输出文件夹中找到对应的结果文件：")
        for filename in sorted(missing_output_files):
            print(f"- {filename}.csv")
    if incomplete_processing_files:
        print("\n⚠️ 以下文件可能未完全处理（输出文件评论ID不完整或输出文件读取失败）：")
        for filename_info in sorted(incomplete_processing_files):
            print(f"- {filename_info}")
    print("\n请重新运行主流程单元格 (cell_id: new_cell_20) 来处理这些文件。")

In [ ]:
from pathlib import Path
import re
import pandas as pd
from datetime import datetime
import hashlib
import time
import random
import sqlite3
import json

from google import genai
from google.genai import types

# =========================
# 2. API configuration
# =========================
PROJECT_ID = "api-try-495100"
LOCATION = "us-central1"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

# =========================
# 3. Performance and reliability settings
# =========================
MIN_INTERVAL_SEC = 3.0
_last_call = 0.0

WRITE_EVERY = 30
TEXT_MAX_CHARS = 900
MAX_RETRIES = 5
MAX_TOTAL_SEC_PER_ITEM = 90
MIN_CONFIDENCE_FOR_FALLBACK = 0.6

# =========================
# 4. Four park experience categories
# =========================
PERCEPTION_CLASSES = [
    "Accessibility",
    "Safety",
    "Usability",
    "Comfort"
]

PERCEPTION_DATA_COLS = []
for c in PERCEPTION_CLASSES:
    PERCEPTION_DATA_COLS.append(f"EV_{c}")
    PERCEPTION_DATA_COLS.append(f"SCORE_{c}")

# =========================
# 5. Lexicon
# =========================
LEXICON = {
    "Accessibility": [
        "交通方便", "交通便利", "方便到达", "好找", "位置好", "位置方便", "地铁", "公交",
        "停车方便", "入口", "出入口", "步行", "路线", "路很好走", "通达", "可达",
        "不好找", "难找", "太远", "远", "交通不便", "不方便", "绕路", "堵车",
        "停车难", "入口难找", "路不好走", "爬坡", "台阶多", "不通", "封路"
    ],
    "Safety": [
        "安全", "安心", "放心", "秩序好", "管理好", "保安", "巡逻", "灯光好", "照明好",
        "人多安全", "适合晚上", "不安全", "危险", "害怕", "吓人", "吓死", "没人",
        "偏僻", "黑", "太黑", "没灯", "照明差", "混乱", "滑", "摔倒", "危险路段",
        "治安", "偷", "骗子", "小偷"
    ],
    "Usability": [
        "设施齐全", "设施完善", "厕所", "卫生间", "座椅", "长椅", "儿童乐园",
        "游乐设施", "健身器材", "运动场", "球场", "跑道", "步道", "休息区",
        "服务好", "管理好", "维护好", "干净", "适合带孩子", "适合老人", "遛娃",
        "遛狗", "野餐", "露营", "设施少", "没厕所", "厕所少", "座椅少",
        "维护差", "破旧", "脏乱", "不方便使用", "没地方坐", "功能少", "不好用"
    ],
    "Comfort": [
        "舒服", "舒适", "惬意", "放松", "安静", "清静", "清幽", "幽静", "凉快",
        "凉爽", "阴凉", "树荫", "空气好", "空气清新", "环境好", "干净", "整洁",
        "适合散步", "适合休闲", "人少", "不挤", "很吵", "嘈杂", "太晒", "暴晒",
        "闷热", "热", "蚊子", "脏", "垃圾", "异味", "臭", "拥挤", "人太多",
        "体验差", "不舒服", "累", "冷", "风大"
    ]
}

def lexicon_hint_labels(text: str):
    t = text or ""
    hits = set()
    for lab, words in LEXICON.items():
        if any(w in t for w in words):
            hits.add(lab)
    return [c for c in PERCEPTION_CLASSES if c in hits]

def evidence_from_text(label: str, text: str, max_len: int = 80) -> str:
    t = text or ""
    words = LEXICON.get(label, [])
    for w in words:
        idx = t.find(w)
        if idx != -1:
            return t[max(0, idx-20):min(len(t), idx+len(w)+20)].strip()
    return t[:max_len].strip()

def fallback_score(label: str, text: str) -> int:
    t = text or ""

    negative_words = [
        "不方便", "不好找", "难找", "太远", "远", "交通不便", "停车难", "入口难找",
        "不安全", "危险", "害怕", "吓人", "吓死", "太黑", "没灯", "照明差", "混乱",
        "设施少", "没厕所", "厕所少", "座椅少", "维护差", "破旧", "脏乱", "不好用",
        "很吵", "嘈杂", "太晒", "暴晒", "闷热", "蚊子", "脏", "垃圾", "异味",
        "臭", "拥挤", "人太多", "体验差", "不舒服"
    ]

    if any(w in t for w in negative_words):
        return -1
    return 1

# =========================
# ⑥ Prompt
# =========================
SYSTEM = (
    "你是中文公园游客评论的多维感知标注员。\n"
    "你的任务是判断评论中是否涉及以下四类公园感知指标，并给出方向性评分：\n\n"
    "1. Accessibility：可通行度，指到达公园及在公园内部通行的便利程度，包括交通便利性、入口可达性、路径清晰度、步行友好度、停车便利性、是否存在通行障碍等。\n"
    "2. Safety：安全度，指用户对公园环境的安全感，包括治安、照明、人流秩序、道路或设施安全、是否存在危险因素等。\n"
    "3. Usability：可用性，指公园是否满足实际使用需求，包括厕所、座椅、儿童设施、健身设施、步道、服务、维护、功能适配性等。\n"
    "4. Comfort：舒适性，指游览或停留过程中的身体与心理舒适程度，包括安静程度、拥挤程度、清洁度、空气、遮阴、凉爽、放松感、噪音、异味、炎热等。\n\n"
    "评分标准：\n"
    "1 = 正面评价\n"
    "0 = 未提及 / 无关\n"
    "-1 = 负面评价\n\n"
    "只输出纯文本，不要 JSON，不要解释。\n"
    "输出格式为每行一条：\n"
    "label\tscore\tevidence\n\n"
    "score 只能是 -1、0、1。要求强制输出所有四个维度，未提及的维度输出 0 并且 evidence 留空。\n"
    "最后两行必须是：\n"
    "PRIMARY\t<一个label或空>\n"
    "CONF\t<0到1的小数>\n\n"
    "label 只能从以下列表中选择：\n"
    f"{PERCEPTION_CLASSES}\n\n"
    "evidence 必须来自评论原文，不超过80字，且不要包含制表符和换行。\n"
    "以下是标注示例：\n\n"
    "示例 1\n"
    "评论：\n"
    "湖边虽以结冰，但仍有很多成群的水鸟畅游水中，时时与人互动，鸟群有黑天鹅，鸳鸯，绿头鸭…荟聚其中。 \n"
    "游玩建议：乘公交地铁走北宫门\n"
    "赏花实况：三月来看白玉兰，十月来闻桂花香\n"
    "交通停车：东宫门和新建宫门这边线路有充足的停车场。\n"
    "输出：\n"
    "Accessibility\t1\t乘公交地铁走北宫门\n"
    "Safety\t0\t\n"
    "Usability\t0\t\n"
    "Comfort\t1\t很多成群的水鸟畅游水中，时时与人互动\n"
    "PRIMARY\tAccessibility\n"
    "CONF\t1\n\n"
    "示例 2\n"
    "评论：\n"
    "游玩建议：游玩路线大概4小时左右，路程可能还会走到别的景点。\n"
    "北宫门 → 长廊 → 排云殿 → 佛香阁 → 昆明湖 → 十七孔桥 → 南湖岛 → 东宫门\n"
    "昆明湖可以瞭望到佛香阁等景色，视野宽阔，也是非常出片。\n"
    "交通停车：地铁口在北宫门附近，如果从东宫门结束游玩路线可以公交坐到地铁站，也还是比较方便的\n"
    "输出：\n"
    "Accessibility\t1\t地铁口在北宫门附近\n"
    "Safety\t0\t\n"
    "Usability\t1\t游玩路线大概4小时左右\n"
    "Comfort\t1\t视野宽阔\n"
    "PRIMARY\tAccessibility\n"
    "CONF\t1\n\n"
    "示例 3\n"
    "评论：\n"
    "颐和园的美景太多了，考虑到国庆节期间人很多，且交通不便。我提前攻略了很久最后选择北如意门进入，下午1点半的样子，想当丝滑无排队直接进！进去后的人都不多，走到中间人好多啊，耍到5点30左右天快黑了就赶紧出来了。\n"
    "输出：\n"
    "Accessibility\t-1\t交通不便\n"
    "Safety\t0\t\n"
    "Usability\t0\t\n"
    "Comfort\t-1\t走到中间人好多啊\n"
    "PRIMARY\tAccessibility\n"
    "CONF\t1\n\n"
    "示例 4\n"
    "评论：\n"
    "周六正好赶上冬至，下午特意和家人来到颐和园，欣赏金光穿洞的美景。下午的人流很多，大家都是有目的的前往，为了保证游客的安全这里进行了限流，我们从湖心岛和万寿山不同的角度来看了金光穿洞的美景，冬季的颐和园依旧是如此的美。\n"
    "输出：\n"
    "Accessibility\t0\t\n"
    "Safety\t1\t为了保证游客的安全这里进行了限流\n"
    "Usability\t0\t\n"
    "Comfort\t-1\t下午的人流很多\n"
    "PRIMARY\tSafety\n"
    "CONF\t1\n\n"
    "示例 5\n"
    "评论：\n"
    "目前颐和园植被光秃秃，但秋冬季是松鼠活跃期，无论后山还是西区，哪怕游人如织的主路旁都能看到仍忙碌储食的松鼠。\n"
    "为了过个有安全感的冬天，松鼠大方接纳游客投喂的栗子、粟米棒和花生，只要是带油脂香气的干食，统统来者不拒。\n"
    "有喂出经验的游客已认识常来讨食的几只松鼠，并熟悉它们的领地范围，跟着他们一路找一路喂，能蹭着拍些松鼠进食照。\n"
    "输出：\n"
    "Accessibility\t0\t\n"
    "Safety\t0\t\n"
    "Usability\t0\t\n"
    "Comfort\t1\t秋冬季是松鼠活跃期\n"
    "PRIMARY\tComfort\n"
    "CONF\t1\n\n"
    "示例 6\n"
    "评论：\n"
    "颐和园新建宫门进去，左转看到铜牛，停下拍照。继续走，不管人多与否，赶紧过桥。在人缝里挤来挤去，一定要注意安全！人多的无法想象，都挤在小岛的东侧，找好位置不要动！争取挤到铁栏杆前面。\n"
    "输出：\n"
    "Accessibility\t1\t新建宫门进去\n"
    "Safety\t-1\t一定要注意安全\n"
    "Usability\t0\t\n"
    "Comfort\t-1\t人多的无法想象\n"
    "PRIMARY\tSafety\n"
    "CONF\t1\n\n"
    "示例 7\n"
    "评论：\n"
    "可以准备一天时间去慢慢逛，里面厕所很多。面积很大。\n"
    "输出：\n"
    "Accessibility\t0\t\n"
    "Safety\t0\t\n"
    "Usability\t1\t里面厕所很多\n"
    "Comfort\t1\t慢慢逛\n"
    "PRIMARY\tUsability\n"
    "CONF\t1\n\n"
    "示例 8\n"
    "评论：\n"
    "北京我最爱的竟然是颐和园，有没有很意外和惊喜。前人太懂得享受生活了，在天然的环境里打造一片自己喜欢又舒适的天地何尝不是一种悟境。\n"
    "从紫御湾码头，乘坐游船到达颐和园的南门。背着太阳，乘着微风，漫步在颐和园湖边的小路上，啃着糖葫芦串。悠哉悠哉的，好自在。\n"
    "走到了17孔桥，人开始多了起来。这里的厕所也是爆满了。园区很多地方都在修缮，因为是周一，所以部分景点没有开放。\n"
    "输出：\n"
    "Accessibility\t1\t乘坐游船到达颐和园的南门\n"
    "Safety\t0\t\n"
    "Usability\t-1\t厕所也是爆满了\n"
    "Comfort\t1\t悠哉悠哉的，好自在\n"
    "PRIMARY\tComfort\n"
    "CONF\t1\n\n"
    "示例 9\n"
    "评论：\n"
    "颐和园整体游览体验非常舒适，也很有文化厚度。园区面积大，但路线规划清晰，沿着湖岸、长廊或山体慢慢走，不会觉得杂乱或疲惫。园内整体环境干净，秩序维护得也不错，即使游客较多，也不会显得特别拥挤嘈杂。整体节奏偏慢，适合静心欣赏风景、了解历史，是那种能让人真正放松下来、回味度很高的经典园林景点。\n"
    "输出：\n"
    "Accessibility\t1\t路线规划清晰\n"
    "Safety\t1\t秩序维护得也不错\n"
    "Usability\t1\t适合静心欣赏风景、了解历史\n"
    "Comfort\t1\t整体游览体验非常舒适\n"
    "PRIMARY\tComfort\n"
    "CONF\t1\n\n"
    "示例 10\n"
    "评论：\n"
    "颐和园也去过好几次了，还是夏天去最好看，就是太晒了，暴走真的是累啊～\n"
    "输出：\n"
    "Accessibility\t0\t\n"
    "Safety\t0\t\n"
    "Usability\t0\t\n"
    "Comfort\t-1\t太晒了，暴走真的是累\n"
    "PRIMARY\tComfort\n"
    "CONF\t1\n"
)

def build_user_prompt(text: str, hints: list):
    return (
        "请模仿上述示例，对以下公园评论进行四类感知标注：\n"
        f"评论：{text}\n\n"
        f"词库提示标签（仅作参考，不要机械照抄）：{hints}"
    )

# =========================
# 7. Utility functions
# =========================
def _throttle():
    global _last_call
    now = time.time()
    wait = MIN_INTERVAL_SEC - (now - _last_call)
    if wait > 0:
        time.sleep(wait)
    _last_call = time.time()

def make_comment_id(text: str):
    return hashlib.md5(text.encode("utf-8", errors="ignore")).hexdigest()

def guess_text_col(df: pd.DataFrame):
    candidates = ["评论内容", "comment_detail", "content", "评论", "comment", "review", "text", "内容", "评价"]
    for c in candidates:
        if c in df.columns:
            if df[c].astype(str).str.strip().str.len().sum() > 0:
                return c
    obj_cols = [c for c in df.columns if df[c].dtype == "object" and "图片" not in c and "image" not in c.lower()]
    if not obj_cols:
        return None
    lens = {
        c: df[c].astype(str).map(len).mean()
        for c in obj_cols
        if df[c].astype(str).str.strip().str.len().sum() > 0
    }
    if lens:
        return max(lens, key=lens.get)
    return None

def read_csv_robust(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        return pd.read_csv(path, encoding="gbk")

# =========================
# 8. SQLite cache
# =========================
def init_db(db_path: Path):
    conn = sqlite3.connect(str(db_path))
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    conn.execute("PRAGMA busy_timeout=8000;")
    conn.execute("""
    CREATE TABLE IF NOT EXISTS responses (
      comment_id TEXT PRIMARY KEY,
      status TEXT,
      response_json TEXT,
      updated_at TEXT
    )
    """)
    conn.commit()
    return conn

def cache_set(conn, cid: str, status: str, obj):
    cur = conn.cursor()
    cur.execute("""
      INSERT INTO responses(comment_id,status,response_json,updated_at)
      VALUES(?,?,?,?)
      ON CONFLICT(comment_id) DO UPDATE SET
        status=excluded.status,
        response_json=excluded.response_json,
        updated_at=excluded.updated_at
    """, (
        cid,
        status,
        json.dumps(obj, ensure_ascii=False),
        datetime.now().isoformat(timespec="seconds")
    ))
    conn.commit()
    cur.close()

# =========================
# 9. Core google-genai API call
# =========================
def parse_tsv_output(raw: str):
    raw = (raw or "").strip()
    raw = raw.replace("```tsv", "").replace("```", "").strip()
    lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]

    label_evidence = []
    primary, conf = "", 0.0

    for ln in lines:
        parts = re.split(r"\t+", ln)

        if len(parts) < 2:
            parts = re.split(r"\s{2,}", ln)
            if len(parts) < 2:
                continue

        k = parts[0].strip().upper()

        if k == "PRIMARY":
            primary = parts[1].strip() if len(parts) > 1 else ""

        elif k == "CONF":
            try:
                match = re.search(r"[0-9.]+", parts[1])
                conf = float(match.group()) if match else 0.0
            except Exception:
                conf = 0.0

        elif parts[0].strip() in PERCEPTION_CLASSES:
            label = parts[0].strip()
            score = 1
            evidence = ""

            if len(parts) >= 3:
                try:
                    match = re.search(r"-?1|0", parts[1])
                    score = int(match.group()) if match else 1
                except Exception:
                    score = 1
                evidence = parts[2].strip()

            elif len(parts) == 2:
                evidence = parts[1].strip()

            if score not in [-1, 0, 1]:
                score = 0

            if score != 0:
                label_evidence.append({
                    "label": label,
                    "score": score,
                    "evidence": evidence[:80]
                })

    seen = set()
    cleaned = []
    for item in label_evidence:
        if item["label"] not in seen:
            cleaned.append(item)
            seen.add(item["label"])

    if primary not in PERCEPTION_CLASSES:
        primary = cleaned[0]["label"] if cleaned else ""

    return {
        "label_evidence": cleaned,
        "primary_label": primary,
        "confidence": conf
    }

def call_api(text: str, hints: list) -> dict:
    prompt_text = f"{SYSTEM}\n\n{build_user_prompt(text, hints)}"

    config = types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=800,
        http_options={"timeout": 30000}
    )

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            _throttle()

            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt_text,
                config=config
            )

            raw = response.text
            if not raw:
                raise RuntimeError("Empty Response")

            obj = parse_tsv_output(raw)

            # --- Lexicon fallback logic ---
            if hints and (not obj["label_evidence"] or obj["confidence"] < MIN_CONFIDENCE_FOR_FALLBACK):
                fallback_items = []
                for h in hints:
                    if h in PERCEPTION_CLASSES:
                        fallback_items.append({
                            "label": h,
                            "score": fallback_score(h, text),
                            "evidence": evidence_from_text(h, text)
                        })

                if fallback_items:
                    obj["label_evidence"] = fallback_items
                    obj["primary_label"] = fallback_items[0]["label"]
                    obj["confidence"] = max(obj["confidence"], 0.35)

            return obj

        except Exception as e:
            error_str = str(e)

            if "429" in error_str:
                wait_time = (2 ** (attempt - 1)) * 5 + random.uniform(0, 1)
                print(f"⚠️ 触发限速 (429)，第 {attempt} 次重试，等待 {wait_time:.1f} 秒...")
                time.sleep(wait_time)
            else:
                print(f"❌ Attempt {attempt} failed: {e}，休息 2 秒...")
                time.sleep(2)

            if attempt == MAX_RETRIES:
                print(f"所有 {MAX_RETRIES} 次重试均失败。")
                return {
                    "label_evidence": [],
                    "primary_label": "",
                    "confidence": 0.0,
                    "_error": "ALL_ATTEMPTS_FAILED"
                }

# =========================
# 10. Main workflow
# =========================
INPUT_BASE_DIR = Path(r"/content/drive/MyDrive/CES分析/源数据-重分类/baidumap/香山-植物园2")
OUTPUT_BASE_DIR = Path(r"/content/drive/MyDrive/CES分析/评论-舒适度分析结果/香山-植物园2")

if not INPUT_BASE_DIR.exists():
    raise FileNotFoundError(f"找不到输入文件夹：{INPUT_BASE_DIR}")

OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)
print(f"输出文件夹已准备好：{OUTPUT_BASE_DIR}")

db_path = OUTPUT_BASE_DIR / "_park_perception_cache.sqlite"
conn = None

def flush(rows: list, current_detail_csv: Path, write_header: bool):
    if not rows:
        return

    part = pd.DataFrame(rows)

    base_cols = [
        "original_index",
        "name",
        "user_name",
        "comment_id",
        "text",
        "Perception_labels",
        "Perception_confidence",
        "Primary_label"
    ]

    all_cols = base_cols + PERCEPTION_DATA_COLS + ["status", "updated_at"]

    for c in all_cols:
        if c not in part.columns:
            part[c] = ""

    part = part[all_cols]
    current_detail_csv.parent.mkdir(parents=True, exist_ok=True)
    part.to_csv(
        current_detail_csv,
        mode="a",
        index=False,
        header=write_header,
        encoding="utf-8-sig"
    )

def get_timestamp_from_filename(filepath: Path) -> datetime:
    match = re.search(r"-(\d{8}_\d{4})\.csv$", filepath.name)
    if match:
        return datetime.strptime(match.group(1), "%Y%m%d_%H%M")
    return datetime.min

all_input_csv_files = list(INPUT_BASE_DIR.glob("**/*.csv"))

if not all_input_csv_files:
    print(f"在 {INPUT_BASE_DIR} 及其子文件夹中未找到任何 CSV 文件。")
else:
    print(f"找到 {len(all_input_csv_files)} 个 CSV 文件进行处理。")

total_processed_files = 0

try:
    conn = init_db(db_path)

    for file_idx, input_file_path in enumerate(all_input_csv_files):
        print(f"\n--- 处理文件 {file_idx+1}/{len(all_input_csv_files)}: {input_file_path.name} ---")

        sub_attraction_name = (
            input_file_path.stem
            if input_file_path.parent == INPUT_BASE_DIR
            else input_file_path.parent.name
        )

        current_detail_csv = None
        write_header = True

        try:
            df = read_csv_robust(input_file_path)
            text_col = guess_text_col(df)

            if text_col is None:
                print(f"警告: 文件 {input_file_path.name} 中未找到合适的文本列，跳过。")
                continue

            expected_cids_from_input = set()
            for text_content in df[text_col].dropna().astype(str):
                if text_content.strip():
                    expected_cids_from_input.add(make_comment_id(text_content.strip()))

            matching_output_files = []
            for existing_output_candidate in OUTPUT_BASE_DIR.glob("*.csv"):
                match_output_name = re.match(
                    r"^(.*?)-(\d{8}_\d{4})\.csv$",
                    existing_output_candidate.name
                )
                if match_output_name and match_output_name.group(1) == sub_attraction_name:
                    matching_output_files.append(existing_output_candidate)

            actual_cids_in_output = set()

            if matching_output_files:
                latest_output_csv = max(matching_output_files, key=get_timestamp_from_filename)

                try:
                    existing_output_df = read_csv_robust(latest_output_csv)

                    if "comment_id" in existing_output_df.columns:
                        actual_cids_in_output = set(
                            existing_output_df["comment_id"].dropna().astype(str).tolist()
                        )
                    else:
                        actual_cids_in_output = set()

                    if (
                        expected_cids_from_input.issubset(actual_cids_in_output)
                        and len(actual_cids_in_output) >= len(expected_cids_from_input)
                    ):
                        print(f"文件 {latest_output_csv.name} 已完整处理，跳过。")
                        total_processed_files += 1
                        continue
                    else:
                        current_detail_csv = latest_output_csv
                        write_header = False
                        print(f"检测到未完成的输出文件: {latest_output_csv.name}，将继续处理。")

                except Exception as e:
                    print(f"警告: 读取现有输出文件 {latest_output_csv.name} 失败 ({e})，将创建新文件。")

            if current_detail_csv is None:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M")
                current_detail_csv = OUTPUT_BASE_DIR / f"{sub_attraction_name}-{timestamp}.csv"
                print(f"未检测到有效输出文件: {current_detail_csv.name}")

            buffer = []
            processed_items_count = 0
            total_comments = len(df)

            for idx, row_data in df.iterrows():
                text = str(row_data.get(text_col, "")).strip()

                if not text:
                    continue

                cid = make_comment_id(text)

                if cid in actual_cids_in_output:
                    processed_items_count += 1
                    continue

                text_for_model = text[:TEXT_MAX_CHARS]
                hints = lexicon_hint_labels(text_for_model)

                print(
                    f"[{file_idx+1}/{len(all_input_csv_files)}] "
                    f"文件 [{input_file_path.name}] - 正在处理评论 "
                    f"{processed_items_count+1}/{total_comments}..."
                )

                processed_items_count += 1

                res = call_api(text_for_model, hints)
                le = res.get("label_evidence", [])

                label_info = {
                    item["label"]: item
                    for item in le
                    if "label" in item
                }

                output_row = {
                    "original_index": idx,
                    "name": row_data.get("name", ""),
                    "user_name": row_data.get("user_name", ""),
                    "comment_id": cid,
                    "text": text_for_model,
                    "Perception_labels": ";".join(label_info.keys()),
                    "Perception_confidence": res.get("confidence", 0.0),
                    "Primary_label": res.get("primary_label", ""),
                    "status": "ok",
                    "updated_at": datetime.now().isoformat(timespec="seconds")
                }

                for c in PERCEPTION_CLASSES:
                    if c in label_info:
                        output_row[f"EV_{c}"] = label_info[c].get("evidence", "")
                        output_row[f"SCORE_{c}"] = label_info[c].get("score", 0)
                    else:
                        output_row[f"EV_{c}"] = ""
                        output_row[f"SCORE_{c}"] = 0

                buffer.append(output_row)
                cache_set(conn, cid, "ok", res)

                print("😴 强制休息 2 秒...")
                time.sleep(2)

                if len(buffer) >= WRITE_EVERY:
                    flush(buffer, current_detail_csv, write_header)
                    buffer = []
                    write_header = False

            if buffer:
                flush(buffer, current_detail_csv, write_header)

            print(f"文件 {input_file_path.name} 处理完成！结果保存至: {current_detail_csv}")
            total_processed_files += 1

        except Exception as e:
            print(f"处理文件 {input_file_path.name} 时发生错误: {e}")

finally:
    if conn:
        try:
            conn.close()
            time.sleep(0.1)
        except Exception as close_e:
            print(f"Error closing DB: {close_e}")

print(f"\n所有文件处理完毕！共处理 {total_processed_files} 个文件。")
